# Hint Generation Experiments
This notebook contains the experiments for the hint generation section of the report.

### Setup

In [1]:
from utils import llm, parse_json
from run import Run
from metrics import RobustnessMetric, CorrectnessMetric, QualityMetric
import hint_prompts
import csv
import json
import os

### Input Context Evaluation
Input Contexts are the various piecies of information used as input by the LLM to generate a hint. The basic input contexts are the question, and student's solution. Additional contexts include the model solution and test case results. In this section we explore which combination of input contexts leads to the best robustness and correctness.


In [7]:
QUESTIONS = ["broken_firewall", "closest_starsystems", "busy_moon_rovers", "purchase_tickets"]
MODELS = ['o1', 'gpt-4o']
QUESTION_FILES = ['question.md', 'question_summary.md']
ITERATIONS = 3 
PROMPTS = [
    {
        'name': 'basic_context',
        'prompt': hint_prompts.exp1_basic_context_prompt,
        'test_case_required': False
    },
    {
        'name': 'test_case_reduced',
        'prompt': hint_prompts.exp1_test_case_prompt,
        'test_case_required': True,
        'test_case_mode': 'REDUCED'
    },
    {
        'name': 'test_case_full',
        'prompt': hint_prompts.exp1_test_case_prompt,
        'test_case_required': True,
        'test_case_mode': 'FULL'
    }
]
results = []


csv_file = "experiment_results_2.csv"
fieldnames = ["model", "question", "question_file", "solution", "solution_type",
              "prompt_type", "test_case_mode", "iteration",
              "prompt", "response", "robustness", "correctness_score", 
              "correctness_justification", "expected_hint"]

# Read existing results to skip completed experiments
existing_results = set()
if os.path.exists(csv_file):
    with open(csv_file, 'r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Create a unique key for each experiment
            experiment_key = (
                row['model'], row['question'], row['question_file'],
                row['solution'], row['prompt_type'], int(row['iteration'])
            )
            existing_results.add(experiment_key)

# Open CSV file in append mode
csv_writer = csv.DictWriter(open(csv_file, 'a', newline='') if os.path.exists(csv_file) else csv.DictWriter(open(csv_file, 'w', newline='')), fieldnames=fieldnames)

# Write header only if file is new
if not os.path.exists(csv_file):
    csv_writer.writeheader()


for model in MODELS:
    for question in QUESTIONS:
        directory = f"questions/{question}"
        for question_file_name in QUESTION_FILES:
            question_file = f"{directory}/{question_file_name}"
        
            # Load expected hints for correctness evaluation
            with open(f"{directory}/expected_hints.json", "r", encoding="utf-8") as f:
                expected_hints = json.load(f)
            
            # Define all solution types
            robustness_solutions = [(f"{directory}/{i}_correct.py", f"{directory}/{i}_correct.json", None) for i in range(1, 4)]
            incomplete_solutions = [(f"{directory}/{i}_incomplete.py", f"{directory}/{i}_incomplete.json", f"{i}_incomplete") for i in range(1, 4)]
            logical_solutions = [(f"{directory}/{i}_logical.py", f"{directory}/{i}_logical.json", f"{i}_logical") for i in range(1, 4)]
            
            all_solutions = (
                robustness_solutions +  # These only need robustness evaluation
                incomplete_solutions +   # These only need correctness evaluation
                logical_solutions       # These only need correctness evaluation
            )
            
            for solution in all_solutions:
                solution_file, test_case_file, hint_key = solution
                
                for prompt_config in PROMPTS:
                    print(f'working on: {solution_file} with model: {model} and prompt: {prompt_config["name"]}')
                    for iteration in range(ITERATIONS):
                        # Check if this experiment has already been done
                        experiment_key = (
                            model, question, question_file_name,
                            solution_file, prompt_config['name'], iteration + 1
                        )
                        
                        if experiment_key in existing_results:
                            print(f'  Skipping iteration {iteration + 1}/{ITERATIONS} (already done)')
                            continue


                        print(f'  iteration {iteration + 1}/{ITERATIONS}')

                        if prompt_config['test_case_required']:
                            run = Run(question_file, solution_file, test_case_file, 
                                    test_case_result_mode=prompt_config['test_case_mode'])
                        else:
                            run = Run(question_file, solution_file)
                        
                        prompt = prompt_config['prompt'] + '\n' + run.__str__()
                        response = llm(prompt, model=model)
                        hint = parse_json(response, "hint")
                        
                        # Initialize metrics as None
                        robustness_score = None
                        correctness_score = None
                        correctness_justification = None
                        expected_hint = None
                        
                        if hint_key is None:  # This is a complete solution, evaluate robustness
                            robustness_metric = RobustnessMetric(code=run.studentSolution, hint=hint)
                            robustness_score = robustness_metric.evaluate()
                        else:  # This is an incomplete or logical solution, evaluate correctness
                            expected_hint = expected_hints[hint_key]
                            correctness_metric = CorrectnessMetric(
                                code=run.studentSolution,
                                hint=hint,
                                expected_hint=expected_hint
                            )
                            correctness_score, correctness_justification = correctness_metric.evaluate()

                        result = {
                            "model": model,
                            "question": question,
                            "question_file": question_file_name,
                            "solution": solution_file,
                            "solution_type": "correct" if hint_key is None else hint_key.split('_')[1],
                            "prompt_type": prompt_config['name'],
                            "test_case_mode": prompt_config.get('test_case_mode', None),
                            "iteration": iteration + 1,
                            "prompt": prompt,
                            "response": response,
                            "robustness": robustness_score,
                            "correctness_score": correctness_score,
                            "correctness_justification": correctness_justification,
                            "expected_hint": expected_hint
                        }
                        csv_writer.writerow(result)                     
                             

print(f"Results successfully written to {csv_file}")

working on: questions/broken_firewall/1_correct.py with model: o1 and prompt: basic_context
  Skipping iteration 1/3 (already done)
  Skipping iteration 2/3 (already done)
  Skipping iteration 3/3 (already done)
working on: questions/broken_firewall/1_correct.py with model: o1 and prompt: test_case_reduced
  Skipping iteration 1/3 (already done)
  Skipping iteration 2/3 (already done)
  Skipping iteration 3/3 (already done)
working on: questions/broken_firewall/1_correct.py with model: o1 and prompt: test_case_full
  Skipping iteration 1/3 (already done)
  Skipping iteration 2/3 (already done)
  Skipping iteration 3/3 (already done)
working on: questions/broken_firewall/2_correct.py with model: o1 and prompt: basic_context
  Skipping iteration 1/3 (already done)
  Skipping iteration 2/3 (already done)
  Skipping iteration 3/3 (already done)
working on: questions/broken_firewall/2_correct.py with model: o1 and prompt: test_case_reduced
  Skipping iteration 1/3 (already done)
  Skipping 

### Prompting Approach Evaluation

In [2]:
# METRICS: Correctness, Detail, LevelOfDetail, Specificty
QUESTIONS = ["broken_firewall", "closest_starsystems", "busy_moon_rovers", "purchase_tickets"]
QUESTION_FILES = ['question_summary.md']
ITERATIONS = 3
APPROACHES = [
    {
        'name': 'claude-3-5-sonnet-cot',
        'model': 'claude-3-5-sonnet-20241022',
        'prompt': hint_prompts.final_prompt_cot,
        'test_case_required': True,
        'test_case_mode': 'FULL'
    },
    {
        'name': 'claude-3-5-sonnet-cot-and-reflection',
        'model': 'claude-3-5-sonnet-20241022',
        'prompt': hint_prompts.final_prompt_cot_and_reflection,
        'test_case_required': True,
        'test_case_mode': 'FULL'
    },
    {
        'name': 'claude-3-5-sonnet-cot-w-explicit-steps',
        'model': 'claude-3-5-sonnet-20241022',
        'prompt': hint_prompts.final_prompt_cot_w_explicit_steps,
        'test_case_required': True,
        'test_case_mode': 'FULL'
    },
    {
        'name': 'gpt-4o-mini-cot-w-explicit-steps',
        'model': 'gpt-4o-mini',
        'prompt': hint_prompts.final_prompt_cot_w_explicit_steps,
        'test_case_required': True,
        'test_case_mode': 'FULL'
    },
    {
        'name': 'gpt-4o-cot',
        'model': 'gpt-4o',
        'prompt': hint_prompts.final_prompt_cot,
        'test_case_required': True,
        'test_case_mode': 'FULL'
    },
    {
        'name': 'gpt-4o-cot-and-reflection',
        'model': 'gpt-4o',
        'prompt': hint_prompts.final_prompt_cot_and_reflection,
        'test_case_required': True,
        'test_case_mode': 'FULL'
    },
    {
        'name': 'gpt-4o-cot-w-explicit-steps',
        'model': 'gpt-4o',
        'prompt': hint_prompts.final_prompt_cot_w_explicit_steps,
        'test_case_required': True,
        'test_case_mode': 'FULL'
    },
    {
        'name': 'o3-mini-no-cot',
        'model': 'o3-mini',
        'prompt': hint_prompts.final_prompt_no_cot,
        'test_case_required': True,
        'test_case_mode': 'FULL'
    },
    # Add more approach configurations as needed
]
results = []


csv_file = "experiment_results_prompting.csv"
fieldnames = ["model", "question", "question_file", "solution", "solution_type",
              "prompt_type", "test_case_mode", "iteration",
              "prompt", "response", 
              "input_tokens", "input_words",
              "output_tokens", "output_words",
              "time_taken",
              "correctness_score", "correctness_justification",
              "specificity_score", "specificity_justification",
              "level_of_detail_score", "level_of_detail_justification",
              "expected_hint"]

# First check if file exists before opening
file_exists = os.path.exists(csv_file)

# Read existing results to skip completed experiments
existing_results = set()
if file_exists:
    with open(csv_file, 'r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Create a unique key for each experiment
            experiment_key = (
                row['model'], row['question'], row['question_file'],
                row['solution'], row['prompt_type'], int(row['iteration'])
            )
            existing_results.add(experiment_key)

# Open CSV file and handle writing
with open(csv_file, 'a' if file_exists else 'w', newline='') as f:
    csv_writer = csv.DictWriter(f, fieldnames=fieldnames)
    
    # Write header only if file is new
    if not file_exists:
        csv_writer.writeheader()


    for question in QUESTIONS:
        directory = f"questions/{question}"
        for question_file_name in QUESTION_FILES:
            question_file = f"{directory}/{question_file_name}"
        
            # Load expected hints for correctness evaluation
            with open(f"{directory}/expected_hints.json", "r", encoding="utf-8") as f:
                expected_hints = json.load(f)
            
            # Define all solution types
            incomplete_solutions = [(f"{directory}/{i}_incomplete.py", f"{directory}/{i}_incomplete.json", f"{i}_incomplete") for i in range(1, 4)]
            logical_solutions = [(f"{directory}/{i}_logical.py", f"{directory}/{i}_logical.json", f"{i}_logical") for i in range(1, 4)]
            
            all_solutions = incomplete_solutions + logical_solutions
            
            for solution in all_solutions:
                solution_file, test_case_file, hint_key = solution
                
                for approach in APPROACHES:
                    print(f'working on: {solution_file} with approach: {approach["name"]}')
                    for iteration in range(ITERATIONS):
                        # Check if this experiment has already been done
                        experiment_key = (
                            approach['model'], question, question_file_name,
                            solution_file, approach['name'], iteration + 1
                        )
                        
                        if experiment_key in existing_results:
                            print(f'  Skipping iteration {iteration + 1}/{ITERATIONS} (already done)')
                            continue

                        print(f'  iteration {iteration + 1}/{ITERATIONS}')

                        if approach['test_case_required']:
                            run = Run(question_file, solution_file, test_case_file, 
                                    test_case_result_mode=approach.get('test_case_mode'))
                        else:
                            run = Run(question_file, solution_file)
                        
                        prompt = approach['prompt'] + '\n' + run.__str__()
                        response, input_metrics, output_metrics, time_taken = llm(
                            prompt, 
                            model=approach['model'],
                            diagnostics=True
                        )
                        hint = parse_json(response, "hint")
                        
                        # Evaluate all quality metrics at once
                        expected_hint = expected_hints[hint_key]
                        quality_metric = QualityMetric(
                            code=run.studentSolution,
                            hint=hint,
                            expected_hint=expected_hint
                        )
                        (correctness_score, correctness_justification,
                        specificity_score, specificity_justification,
                        level_of_detail_score, level_of_detail_justification) = quality_metric.evaluate()

                        result = {
                            "model": approach['model'],
                            "question": question,
                            "question_file": question_file_name,
                            "solution": solution_file,
                            "solution_type": hint_key.split('_')[1],
                            "prompt_type": approach['name'],
                            "test_case_mode": approach.get('test_case_mode', None),
                            "iteration": iteration + 1,
                            "prompt": prompt,
                            "response": response,
                            "input_tokens": input_metrics.get('tokens'),
                            "input_words": input_metrics['words'],
                            "output_tokens": output_metrics.get('tokens'),
                            "output_words": output_metrics['words'],
                            "time_taken": time_taken,
                            "correctness_score": correctness_score,
                            "correctness_justification": correctness_justification,
                            "specificity_score": specificity_score,
                            "specificity_justification": specificity_justification,
                            "level_of_detail_score": level_of_detail_score,
                            "level_of_detail_justification": level_of_detail_justification,
                            "expected_hint": expected_hint
                        }
                        csv_writer.writerow(result)          
                             

print(f"Results successfully written to {csv_file}")

working on: questions/broken_firewall/1_incomplete.py with approach: claude-3-5-sonnet-cot
  iteration 1/3
  iteration 2/3
  iteration 3/3
working on: questions/broken_firewall/1_incomplete.py with approach: claude-3-5-sonnet-cot-and-reflection
  iteration 1/3
  iteration 2/3
  iteration 3/3
working on: questions/broken_firewall/1_incomplete.py with approach: claude-3-5-sonnet-cot-w-explicit-steps
  iteration 1/3
  iteration 2/3
  iteration 3/3
working on: questions/broken_firewall/1_incomplete.py with approach: gpt-4o-mini-cot-w-explicit-steps
  iteration 1/3
  iteration 2/3
  iteration 3/3
working on: questions/broken_firewall/1_incomplete.py with approach: gpt-4o-cot
  iteration 1/3
  iteration 2/3
  iteration 3/3
working on: questions/broken_firewall/1_incomplete.py with approach: gpt-4o-cot-and-reflection
  iteration 1/3
  iteration 2/3
  iteration 3/3
working on: questions/broken_firewall/1_incomplete.py with approach: gpt-4o-cot-w-explicit-steps
  iteration 1/3
  iteration 2/3
 